# EML Network Learning Sine Function

This notebook demonstrates an EML (Exponential-Minus-Logarithm) neural network learning to approximate a sine function from 100 randomly sampled datapoints.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random
from micrograd.engine import Value
from micrograd.nn import EMLMLP

## Generate Training Data

Create 100 random datapoints sampled from a sine function.

In [ ]:
# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# Generate 100 datapoints from sine function
n_samples = 100
x_data = np.random.uniform(-2*np.pi, 2*np.pi, n_samples)
y_data = np.sin(x_data)

# Visualize the training data
plt.figure(figsize=(10, 4))
plt.scatter(x_data, y_data, alpha=0.6, s=30)
plt.xlabel('x')
plt.ylabel('sin(x)')
plt.title('Training Data: 100 samples from sine function')
plt.grid(True, alpha=0.3)
plt.show()

## Build and Train the EML Network

Create an EMLMLP with an input layer (1 neuron for x input), hidden layers, and 1 output neuron for the sine value.

In [ ]:
# Create EML MLP: 1 input -> 16 hidden -> 16 hidden -> 1 output
model = EMLMLP(1, [16, 16, 1])

print(f"Model architecture:")
for layer in model.layers:
    print(f"  {layer}")
    
# Count parameters
n_params = len(model.parameters())
print(f"\nTotal parameters: {n_params}")

## Training Loop

Train the network using gradient descent to minimize mean squared error.

In [ ]:
# Training parameters
learning_rate = 0.01
epochs = 300

# Store loss history
loss_history = []

# Training loop
for epoch in range(epochs):
    # Forward pass
    total_loss = Value(0.)
    
    for xi, yi in zip(x_data, y_data):
        # Forward pass for single sample
        x_input = [Value(xi)]
        y_pred = model(x_input)
        
        # MSE loss for this sample
        loss = (y_pred - Value(yi)) ** 2.
        total_loss = total_loss + loss
    
    # Average loss
    avg_loss = total_loss * Value(1.0 / n_samples)
    loss_history.append(float(avg_loss.data))
    
    # Backward pass
    model.zero_grad()
    avg_loss.backward()
    
    # Update parameters
    for p in model.parameters():
        if isinstance(p.data, complex):
            # For complex parameters, update real and imaginary parts
            p.data = p.data - learning_rate * p.grad
        else:
            p.data = p.data - learning_rate * p.grad
    
    # Print progress
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {loss_history[-1]:.6f}")

print(f"\nTraining complete! Final loss: {loss_history[-1]:.6f}")

## Visualize Training Loss

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(loss_history, linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.title('Training Loss Over Time')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

## Evaluate on Test Set

Evaluate the trained model on the original training data and compare with the true sine function.

In [ ]:
# Generate evaluation points
x_eval = np.linspace(-2*np.pi, 2*np.pi, 200)
y_eval_true = np.sin(x_eval)
y_eval_pred = []

# Predict on evaluation set
for x_val in x_eval:
    x_input = [Value(x_val)]
    y_pred = model(x_input)
    
    # Extract real value
    if isinstance(y_pred.data, complex):
        y_eval_pred.append(float(y_pred.data.real))
    else:
        y_eval_pred.append(float(y_pred.data))

y_eval_pred = np.array(y_eval_pred)

# Calculate MSE on evaluation set
mse = np.mean((y_eval_pred - y_eval_true) ** 2)
print(f"Mean Squared Error on evaluation set: {mse:.6f}")

## Visualization: Model Prediction vs True Sine Function

In [ ]:
plt.figure(figsize=(12, 6))

# Plot true sine function
plt.plot(x_eval, y_eval_true, 'b-', linewidth=2, label='True sin(x)', alpha=0.7)

# Plot model prediction
plt.plot(x_eval, y_eval_pred, 'r--', linewidth=2, label='EML Network Prediction', alpha=0.7)

# Plot training data
plt.scatter(x_data, y_data, alpha=0.4, s=30, label='Training Data', color='green')

plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('EML Network Learning Sine Function (100 Training Samples)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xlim(-2*np.pi, 2*np.pi)
plt.ylim(-1.5, 1.5)
plt.show()

print(f"Visualization complete!")

## Residuals Analysis

Analyze the prediction errors.

In [ ]:
residuals = y_eval_pred - y_eval_true

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals over x
axes[0].plot(x_eval, residuals, 'b-', alpha=0.7, linewidth=2)
axes[0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
axes[0].set_xlabel('x')
axes[0].set_ylabel('Residual (Predicted - True)')
axes[0].set_title('Prediction Errors Over Input Range')
axes[0].grid(True, alpha=0.3)

# Histogram of residuals
axes[1].hist(residuals, bins=30, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Distribution of Residuals\nMean: {np.mean(residuals):.6f}, Std: {np.std(residuals):.6f}')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Max absolute error: {np.max(np.abs(residuals)):.6f}")